# ADCD external-validation split: condition crops and embeddings

This notebook parses the ADCD external-validation split (YOLO format, 180 radiographs), draws the parsed boxes on one radiograph for checking, crops the Caries, Impacted and Periapical Lesion boxes, embeds the crops with the frozen Dental ViT-L, and caches the embeddings and labels. `06_threshold_recalibration.ipynb` (Section 8) loads them to compute the ADCD refusal rates (Table 9) and the ADCD threshold recalibration.

Section 8 optionally embeds the same crops with DINOv3-L.

## 1. Environment and device

In [ ]:
import sys, subprocess
def pipq(*p): subprocess.run([sys.executable,"-m","pip","install","-q","-U",*p],check=False)
# Packages are listed in requirements.txt. Uncomment to install them into the active kernel.
# pipq("transformers","huggingface_hub","torch","torchvision","scikit-learn","pillow","tqdm","matplotlib","ml_collections")
import torch, numpy as np, os
from PIL import Image
DEVICE="mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

## 2. Label space and paths

ADCD (Zenodo record 15487430) is in YOLO format with 14 classes. Set `ADCD_ROOT` to the dataset folder and `ADCD_DIR` to its external-validation split (180 radiographs from other clinics). Class indices from the dataset's `data.yaml`: 0 IMP, 1 PRR, 2 OBT, 3 END, 4 CAR, 5 BON, 6 IMT, 7 API, 8 ROT, 9 FUR, 10 APS, 11 ROR, 12 ORD, 13 SRD. CAR (4), IMT (6) and API (7) are mapped to Caries, Impacted and Periapical Lesion. ADCD has no separate Deep Caries class.

In [ ]:
# Label space, paths, cache folder and crop helper
import glob
DX_NAMES = ["Impacted", "Caries", "Periapical Lesion", "Deep Caries"]   # DENTEX order
ADCD_ROOT = "/path/to/ADCD/panoramic_radiography_yolo_dataset_14_classes"
ADCD_DIR  = os.path.join(ADCD_ROOT, "EXTERNAL_VALIDATION_FOLDER")   # external-validation split (180 radiographs)
CACHE = os.path.join(ADCD_ROOT, "_cache"); os.makedirs(CACHE, exist_ok=True)

def crop_box(img, box, pad=0.15, min_size=8):
    W, H = img.size; x0, y0, x1, y1 = box
    x0, x1 = min(x0, x1), max(x0, x1); y0, y1 = min(y0, y1), max(y0, y1); w, h = x1 - x0, y1 - y0
    px, py = w*pad, h*pad
    cx0, cy0 = max(0, int(x0 - px)), max(0, int(y0 - py)); cx1, cy1 = min(W, int(x1 + px)), min(H, int(y1 + py))
    if cx1 - cx0 < min_size: cx1 = min(W, cx0 + min_size)
    if cy1 - cy0 < min_size: cy1 = min(H, cy0 + min_size)
    if cx1 <= cx0 or cy1 <= cy0: return None
    return img.crop((cx0, cy0, cx1, cy1))
print("ADCD_DIR:", ADCD_DIR, "| cache:", CACHE)

## 3. Image and label folders

In [ ]:
# Image and label folders, and one label file
def _find(root, kind):
    return [d for d,_,fs in os.walk(root) if os.path.basename(d).lower()==kind and fs]
print("image folders:", _find(ADCD_DIR,"images"))
print("label folders:", _find(ADCD_DIR,"labels"))
lbls=[]
for d in _find(ADCD_DIR,"labels"):
    lbls += glob.glob(os.path.join(d,"*.txt"))
print("total label files:", len(lbls))
if lbls:
    print("--- sample", os.path.basename(lbls[0]), "---")
    print(open(lbls[0]).read()[:300])

## 4. YOLO parser

In [ ]:
# YOLO parser, mapped to the DENTEX label space
ADCD_CLASS_TO_DX = {4:"Caries", 6:"Impacted", 7:"Periapical Lesion"}

def _img_for(label_path):
    stem=os.path.splitext(os.path.basename(label_path))[0]
    for d,_,fs in os.walk(ADCD_DIR):
        if os.path.basename(d).lower()!="images": continue
        for ext in (".jpg",".jpeg",".png",".JPG",".PNG"):
            p=os.path.join(d, stem+ext)
            if os.path.exists(p): return p
    hits=[]
    for d,_,fs in os.walk(ADCD_DIR):
        hits += glob.glob(os.path.join(d, stem+".*"))
    return hits[0] if hits else None

def parse_adcd():
    NUM = len(DX_NAMES)
    label_files=[]
    for d,_,fs in os.walk(ADCD_DIR):
        if os.path.basename(d).lower()=="labels":
            label_files += glob.glob(os.path.join(d,"*.txt"))
    records=[]
    for lf in label_files:
        img_path=_img_for(lf)
        if img_path is None: continue
        W,H = Image.open(img_path).size
        y=np.zeros(NUM,np.float32); boxes=[]
        for line in open(lf):
            parts=line.split()
            if len(parts)<5: continue
            cid=int(float(parts[0])); cx,cy,w,h=map(float,parts[1:5])
            cond=ADCD_CLASS_TO_DX.get(cid)
            if cond is None: continue
            x0=(cx-w/2)*W; y0=(cy-h/2)*H; x1=(cx+w/2)*W; y1=(cy+h/2)*H
            boxes.append([x0,y0,x1,y1]); y[DX_NAMES.index(cond)]=1
        if boxes:
            records.append((os.path.basename(img_path), y, boxes))
    print(f"parsed {len(records)} ADCD images with >=1 relevant condition (of {len(label_files)} label files)")
    return records

## 5. Visual check of the parsed boxes

In [ ]:
# Parsed boxes drawn on one radiograph
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
recs = parse_adcd()
fn, y, boxes = recs[0]
cand=None
for d,_,fs in os.walk(ADCD_DIR):
    if os.path.basename(d).lower()=="images":
        p=os.path.join(d,fn)
        if os.path.exists(p): cand=p; break
if cand is None:
    h=glob.glob(os.path.join(ADCD_DIR,"**",fn),recursive=True); cand=h[0] if h else None
im=Image.open(cand).convert("RGB"); W,H=im.size
fig,ax=plt.subplots(figsize=(13,6)); ax.imshow(im, cmap="gray")
for b in boxes:
    x0,y0,x1,y1=b
    ax.add_patch(Rectangle((x0,y0),x1-x0,y1-y0,fill=False,edgecolor='lime',lw=1.5))
ax.set_title(f"{fn}  {W}x{H}: parsed condition boxes")
print("conditions present (multi-hot over DX_NAMES):", dict(zip(DX_NAMES, y.astype(int))))
plt.tight_layout(); plt.show()

## 6. Frozen Dental ViT-L encoder

In [ ]:
# Frozen Dental ViT-L encoder (defines embed_crop)
import math
if not os.path.isdir("ViT-pytorch"):
    subprocess.run(["git","clone","--depth","1","https://github.com/jeonsworld/ViT-pytorch"],check=True)
sys.path.insert(0,"ViT-pytorch")
from models.modeling import VisionTransformer, CONFIGS
from huggingface_hub import hf_hub_download
import torchvision.transforms as T

ck=hf_hub_download(repo_id="Bryceee/Teeth_Visual_Experts_Models",
    filename="Teeth_Visual_Experts_ViT_L_periapical_images_cls_7diseases.bin")
state=torch.load(ck,map_location="cpu")
if isinstance(state,dict) and "state_dict" in state: state=state["state_dict"]
pos=state["transformer.embeddings.position_embeddings"].shape[1]
IMG=int(round(math.sqrt(pos-1)))*16; nh=state["head.weight"].shape[0]
_m=VisionTransformer(CONFIGS["ViT-L_16"],img_size=IMG,num_classes=nh,zero_head=False)
_m.load_state_dict(state,strict=False); _m=_m.to(DEVICE).eval()
_prep=T.Compose([T.Resize((IMG,IMG)),T.ToTensor(),T.Normalize([0.5]*3,[0.5]*3)])
class _DE(torch.nn.Module):
    def __init__(s,v): super().__init__(); s.v=v
    @torch.no_grad()
    def forward(s,x): seq,_=s.v.transformer(x); return seq[:,0]
_enc=_DE(_m).to(DEVICE).eval()

@torch.no_grad()
def embed_crop(pil):
    x=_prep(pil.convert("RGB")).unsqueeze(0).to(DEVICE)
    return torch.nn.functional.normalize(_enc(x).float().cpu(),dim=1).numpy()[0]
print("dental ViT-L encoder ready @", IMG)

## 7. Crops, embeddings and cache

With `USE_BACKBONE = "dental_vitl"` the embeddings are saved as `emb_adcd_dental_vitl.npy`, the file read by `06_threshold_recalibration.ipynb`. The reported analysis used 513 crops from 136 radiographs.

In [ ]:
# Crop the boxes, embed and cache
from tqdm.auto import tqdm
USE_BACKBONE = "dental_vitl"   # "dinov3" | "dental_vitl"
def embed_fn_batch(pils):
    if USE_BACKBONE=="dinov3": return embed_dinov3_batch(pils, bs=16)
    return np.stack([embed_crop(p) for p in pils]).astype(np.float32)

def _img_path(fn):
    for d,_,fs in os.walk(ADCD_DIR):
        if os.path.basename(d).lower()=="images":
            p=os.path.join(d,fn)
            if os.path.exists(p): return p
    h=glob.glob(os.path.join(ADCD_DIR,"**",fn),recursive=True); return h[0] if h else None

crops=[]; Ys=[]; names=[]; skipped=0
for fn, y, boxes in tqdm(recs):
    cand=_img_path(fn)
    if cand is None: continue
    im=Image.open(cand).convert("RGB"); W,H=im.size
    for b in boxes:
        x0,y0,x1,y1=min(b[0],b[2]),min(b[1],b[3]),max(b[0],b[2]),max(b[1],b[3])
        if (x1-x0)<8 or (y1-y0)<8 or x1<=0 or y1<=0 or x0>=W or y0>=H: skipped+=1; continue
        c=crop_box(im,[x0,y0,x1,y1])
        if c is None: skipped+=1; continue
        crops.append(c); Ys.append(y); names.append(fn)
print(f"collected {len(crops)} ADCD crops from {len(set(names))} radiographs | skipped {skipped}")
X_adcd=embed_fn_batch(crops); Y_adcd=np.stack(Ys); IMG_adcd=np.array(names,dtype=object)
np.save(os.path.join(CACHE, f"emb_adcd_{USE_BACKBONE}.npy"), X_adcd)
np.save(os.path.join(CACHE, "lab_adcd.npy"), Y_adcd)
print("ADCD embeddings:", X_adcd.shape, "| labels:", Y_adcd.shape, "| saved to", CACHE)

## 8. Optional: DINOv3-L embeddings of the same crops

DINOv3-L is a gated model. Accept its licence on the Hugging Face model page and log in once (`huggingface-cli login`, or set the `HF_TOKEN` environment variable) before running. Run the next two cells, set `USE_BACKBONE = "dinov3"` in Section 7 and run Section 7 again. The embeddings are then saved as `emb_adcd_dinov3.npy`.

In [ ]:
# DINOv3 is a gated model on Hugging Face. Accept the license on the model page, then authenticate once
# in this kernel (do not store the token in the notebook):
from huggingface_hub import login
# login()   # prompts for your token; alternatively set the HF_TOKEN environment variable

DINOV3_ID = "facebook/dinov3-vitl16-pretrain-lvd1689m"   # ViT-L/16, 1024-d
from transformers import AutoImageProcessor, AutoModel
dv3_proc  = AutoImageProcessor.from_pretrained(DINOV3_ID)
dv3_model = AutoModel.from_pretrained(DINOV3_ID).to(DEVICE).eval()
print("DINOv3 loaded:", DINOV3_ID, "| hidden size:", dv3_model.config.hidden_size)

In [ ]:
# DINOv3 embedding helpers: pooler_output (image-level embedding), CLS token as fallback, L2-normalised
@torch.no_grad()
def embed_dinov3(pil):
    inp = dv3_proc(images=pil.convert("RGB"), return_tensors="pt").to(DEVICE)
    out = dv3_model(**inp)
    feat = getattr(out, "pooler_output", None)
    if feat is None:
        feat = out.last_hidden_state[:, 0]
    return torch.nn.functional.normalize(feat.float().cpu(), dim=1).numpy()[0]

@torch.no_grad()
def embed_dinov3_batch(pils, bs=16):
    outs=[]
    for i in range(0, len(pils), bs):
        inp = dv3_proc(images=[p.convert("RGB") for p in pils[i:i+bs]], return_tensors="pt").to(DEVICE)
        o = dv3_model(**inp)
        f = getattr(o, "pooler_output", None)
        if f is None: f = o.last_hidden_state[:, 0]
        outs.append(torch.nn.functional.normalize(f.float().cpu(), dim=1).numpy())
    return np.concatenate(outs, 0)
print("DINOv3 embed helpers ready")